# ESG RAG — 03: Evaluation

Checking answers by hand doesn't scale, and it's easy to be fooled by a few good examples.
This notebook runs a set of **114 verified test questions** automatically and scores the RAG.

```
questions.csv → for each question: retrieve → generate (Qwen / Claude) → grade → results table → scores
```

## Metrics

| Metric | Question it answers | How |
|---|---|---|
| **Retrieval hit@k** | Did the search find the right page? | Code: is an expected page among the retrieved chunks? |
| **Answer score** | Is the answer correct? | **LLM-as-a-judge:** Claude compares the answer with a reference answer checked in the PDF |
| **Refusal accuracy** | Does it say "I don't know" when it should? | Judge, on questions the reports can't answer |
| **Citation accuracy** | Does it cite the right page? | Code: do the cited sources include an expected page? |
| **Time and cost** | How fast and how expensive? | Measured during the run |

Retrieval and generation are measured **separately**, which shows *where* each error comes from.

**Requirements:** the databases built by `01_indexing.ipynb`, `questions.csv` in `MyDrive/esg-rag/evals/`,
the Claude API key in Colab Secrets (`ANTHROPIC_API_KEY`, needed for the judge), and a GPU runtime when Qwen is evaluated.

## Step 0 — Settings

- `CONFIG_NAME`: a label for **the experiment**. The results are saved with it, so experiments can be compared later.
- `DB_NAME`: which database to test (`chroma_db`, `chroma_db_vision`, `chroma_db_chunk3000`…), built by `01_indexing.ipynb`.
- `CONTEXT_MODE`: `"chunk"` sends the retrieved chunks to the LLM. `"page"` (**parent page retrieval**) searches with the
  small chunks, then sends the **whole page** of each chunk found, up to `MAX_PAGES` pages.
  `"page_plus_chunks"` sends whole pages for the top `MAX_PAGES` pages **plus** every other retrieved chunk whose page was left out:
  the **depth** of whole pages with the **breadth** of the chunk search.
- `NEIGHBOR_EDGES`: in page mode, also sends the **last chunk of the previous page** and the **first chunk of the next page**,
  for text or tables that continue across pages. They are separate sources, so citations still point to the right page.
- `JUDGE_MODEL`: the model that grades the answers. Haiku is cheap; Sonnet is stricter but costs about 20x more.

In [ ]:
CONFIG_NAME = "final_D_no_vision"   # ← change this label for each experiment (e.g. "final_D_no_vision" with DB_NAME = "chroma_db")

USE_QWEN = True       # evaluate Qwen3-8B (needs the GPU and Ollama)
USE_CLAUDE = True     # evaluate Claude Haiku
N_RESULTS = 10        # how many chunks to retrieve (top-k)
MULTI_COMPANY_MODE = "per_company"   # or "global": how to search questions with no company
DB_NAME = "chroma_db"    # which database from Notebook 01: "chroma_db" (text only) or "chroma_db_vision" (text + image transcriptions)
CONTEXT_MODE = "page_plus_chunks"  # "chunk" | "page": whole pages | "page_plus_chunks": whole top pages + the other chunks
MAX_PAGES = 5                   # for "page" and "page_plus_chunks": maximum number of whole pages sent to the LLM
NEIGHBOR_EDGES = False          # only for "page": also send the END of the previous page and the START of the next page

CLAUDE_MODEL = "claude-haiku-5-5"   # the model being evaluated
JUDGE_MODEL = "claude-haiku-5-5"    # the model that grades; "claude-sonnet-5-5" is stricter but more expensive

LLMS = [name for name, on in [("qwen", USE_QWEN), ("claude", USE_CLAUDE)] if on]
print("Config:", CONFIG_NAME, "| LLMs:", LLMS, "| top-k:", N_RESULTS, "| multi-company:", MULTI_COMPANY_MODE, "| db:", DB_NAME,
      "| context:", CONTEXT_MODE + (f" (max {MAX_PAGES} pages{', + neighbor edges' if NEIGHBOR_EDGES else ''})" if CONTEXT_MODE != "chunk" else ""),
      "| judge:", JUDGE_MODEL)

Config: final_D_no_vision | LLMs: ['qwen', 'claude'] | top-k: 10 | multi-company: per_company | db: chroma_db | context: page_plus_chunks (max 5 pages) | judge: claude-haiku-5-5


## Step 1 — Libraries

In [ ]:
%pip install -q anthropic chromadb sentence-transformers requests pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 r

## Step 2a — 🟢 Set up Qwen with Ollama (only if `USE_QWEN = True`)

Same as `02_ask.ipynb`. It has to run again every new Colab session.

In [ ]:
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"
QWEN_MODEL = "qwen3:8b"

def ollama_is_running():
    try:
        requests.get(OLLAMA_URL, timeout=5)
        return True
    except requests.exceptions.RequestException:
        return False

if USE_QWEN:
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    if not ollama_is_running():
        ollama_log = open("/content/ollama.log", "w")
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_is_running():
            print("Ollama server is running ✅")
            break
        time.sleep(1)
    else:
        print("⚠️ Ollama didn't start. Run: !tail -20 /content/ollama.log")
    !ollama pull {QWEN_MODEL}
else:
    print("Skipping Qwen (USE_QWEN = False)")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server is running ✅


## Step 2b — 🟣 Set up Claude (always needed: it is the judge)

In [ ]:
import anthropic
from google.colab import userdata

client_claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("Claude client ready ✅")

Claude client ready ✅


## Step 3 — Load the database and the embedding model

Same as `02_ask.ipynb`: the database is copied from Drive, and the embedding model is **the same one** used in `01_indexing.ipynb`.

In [ ]:
import os, shutil
import chromadb
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/esg-rag"
DRIVE_DB_DIR = f"{PROJECT_DIR}/{DB_NAME}"
LOCAL_DB_DIR = "/content/chroma_db"
EVALS_DIR = f"{PROJECT_DIR}/evals"
os.makedirs(EVALS_DIR, exist_ok=True)

# If this cell already ran in this session, ChromaDB still holds the old copy open: forget it first
from chromadb.api.client import SharedSystemClient
SharedSystemClient.clear_system_cache()

shutil.rmtree(LOCAL_DB_DIR, ignore_errors=True)
shutil.copytree(DRIVE_DB_DIR, LOCAL_DB_DIR)
db = chromadb.PersistentClient(path=LOCAL_DB_DIR)
collection = db.get_collection("esg_reports")
print("Chunks in the database:", collection.count())

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
print("Embedding model loaded on", device)

Mounted at /content/drive
Chunks in the database: 1369


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded on cuda


## Step 4 — The RAG functions

**The same functions as `02_ask.ipynb`** (retrieval, prompt and generation), without the printing.
⚠️ Any change in `02_ask.ipynb` (for example, the prompt) has to be copied here too, or the evaluation tests the old version.

In [ ]:
import re, time, requests   # needed by the functions below (even when Qwen is off)

def retrieve(question, n_results=5, company=None, year=None):
    filters = []
    if company:
        filters.append({"company": company})
    if year:
        filters.append({"year": year})
    where = None if not filters else (filters[0] if len(filters) == 1 else {"$and": filters})

    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results, where=where)
    return [
        {"text": text, "distance": dist, **meta}
        for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]


# ── Multi-company questions ──────────────────────────────────────────────
# "global": one search over all reports (top-k overall; one company can take most slots)
# "per_company": search each company separately and combine (every company gets a fair share)
COMPANIES = ["natura", "petrobras", "vale"]

def retrieve_per_company(question, n_results=5, year=None, companies=COMPANIES):
    k_per_company = -(-n_results // len(companies))      # rounds up: 5 results / 3 companies -> 2 each
    chunks = []
    for name in companies:
        chunks += retrieve(question, n_results=k_per_company, company=name, year=year)
    return sorted(chunks, key=lambda c: c["distance"])   # most similar first

def smart_retrieve(question, n_results=5, company=None, year=None):
    if company is None and MULTI_COMPANY_MODE == "per_company":
        return retrieve_per_company(question, n_results=n_results, year=year)
    return retrieve(question, n_results=n_results, company=company, year=year)


# ── Parent page retrieval ────────────────────────────────────────────────
# Search with small chunks (precise), but give the LLM the whole page of each chunk found.
_page_cache = {}

def get_page_text(company, year, page):
    '''Rebuilds a page's text from all its chunks in the database (removing the overlaps).'''
    key = (company, year, page)
    if key not in _page_cache:
        res = collection.get(where={"$and": [{"company": company}, {"year": year}, {"page": page}]})
        # IDs look like "petrobras_2025_p147_c2": sort the chunks by their position on the page (c0, c1, c2...)
        parts = sorted(zip(res["ids"], res["documents"]), key=lambda x: int(re.search(r"_c(\d+)$", x[0]).group(1)))
        merged = ""
        for _, text in parts:
            # Each chunk starts with the last ~200 characters of the previous one (the overlap): don't repeat them
            pos = merged.rfind(text[:60], max(0, len(merged) - 600)) if merged else -1
            merged = merged[:pos] + text if pos != -1 else (merged + "\n" + text).strip()
        _page_cache[key] = merged
    return _page_cache[key]


def expand_to_pages(chunks, max_pages=5):
    '''Turns retrieved chunks into whole pages: unique pages, most relevant first,
    alternating between companies so a multi-company question keeps every company.'''
    by_company = {}
    for c in chunks:                                     # chunks are sorted by relevance
        pages = by_company.setdefault(c["company"], [])
        if not any(p["page"] == c["page"] and p["year"] == c["year"] for p in pages):
            pages.append(c)
    selected = []
    while len(selected) < max_pages and any(by_company.values()):
        for company in list(by_company):                # round-robin: one page per company per turn
            if by_company[company] and len(selected) < max_pages:
                selected.append(by_company[company].pop(0))
    pages = [{**c, "text": get_page_text(c["company"], c["year"], c["page"])} for c in selected]
    if not NEIGHBOR_EDGES:
        return pages

    # Add the end of the previous page and the start of the next page (when they aren't already included)
    included = {(p["company"], p["year"], p["page"]) for p in pages}
    result = []
    for p in pages:
        before = get_edge_chunk(p["company"], p["year"], p["page"] - 1, "last")
        if before and (p["company"], p["year"], p["page"] - 1) not in included:
            result.append({**p, "page": p["page"] - 1, "text": before, "note": " (end of page)"})
            included.add((p["company"], p["year"], p["page"] - 1))
        result.append(p)
        after = get_edge_chunk(p["company"], p["year"], p["page"] + 1, "first")
        if after and (p["company"], p["year"], p["page"] + 1) not in included:
            result.append({**p, "page": p["page"] + 1, "text": after, "note": " (start of page)"})
            included.add((p["company"], p["year"], p["page"] + 1))
    return result


def get_edge_chunk(company, year, page, which):
    '''The first or last chunk of a page ("" if the page has no chunks, e.g. a cover or photo page).'''
    if page < 1:
        return ""
    res = collection.get(where={"$and": [{"company": company}, {"year": year}, {"page": page}]})
    if not res["ids"]:
        return ""
    parts = sorted(zip(res["ids"], res["documents"]), key=lambda x: int(re.search(r"_c(\d+)$", x[0]).group(1)))
    return parts[0][1] if which == "first" else parts[-1][1]


def get_context(question, n_results=5, company=None, year=None):
    '''Retrieval used by the evals: chunks, or whole pages if CONTEXT_MODE = "page".'''
    chunks = smart_retrieve(question, n_results=n_results, company=company, year=year)
    if CONTEXT_MODE == "page":
        return expand_to_pages(chunks, max_pages=MAX_PAGES)
    if CONTEXT_MODE == "page_plus_chunks":
        # Depth AND breadth: whole pages for the top MAX_PAGES pages, plus every other retrieved chunk
        # whose page didn't make the cut (so no retrieved information is dropped)
        pages = expand_to_pages(chunks, max_pages=MAX_PAGES)
        page_keys = {(p["company"], p["year"], p["page"]) for p in pages}
        leftovers = [c for c in chunks if (c["company"], c["year"], c["page"]) not in page_keys]
        return pages + leftovers
    return chunks


SYSTEM_PROMPT = '''You are an assistant that answers questions about companies' sustainability reports.

Rules:
1. Use ONLY the information in the sources provided. Do not use outside knowledge.
2. Cite every fact with the source number in brackets, like [1] or [2][3].
3. If the sources don't contain the answer, say clearly that you couldn't find it in the reports. Never guess.
4. When you give numbers, keep the exact units from the source (e.g., tCO2e, m³, %).
5. Answer in the same language as the question.
6. Be concise: a short paragraph or a few bullet points.'''


def build_user_message(question, chunks):
    parts = []
    for i, c in enumerate(chunks, start=1):
        parts.append(f"[{i}] {c['company'].capitalize()} — {c['year']} report, page {c['page']}{c.get('note', '')}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return f"Sources:\n\n{context}\n\n---\n\nQuestion: {question}"


CLAUDE_PRICES = {   # USD per million tokens (input, output)
    "claude-haiku-5-5": (0.10, 0.50),
    "claude-sonnet-5-5": (2.00, 10.00),
}

def claude_cost(model, usage):
    price_in, price_out = CLAUDE_PRICES.get(model, (0, 0))
    return (usage.input_tokens * price_in + usage.output_tokens * price_out) / 1_000_000


# Whole pages are long (up to ~2,000 tokens each), so Qwen needs a bigger window in page mode
QWEN_NUM_CTX = 16384 if CONTEXT_MODE in ("page", "page_plus_chunks") else 8192


def generate_qwen(system_prompt, user_message):
    start = time.time()
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": QWEN_MODEL,
            "messages": [{"role": "system", "content": system_prompt},
                         {"role": "user", "content": user_message}],
            "stream": False,
            "think": False,
            "options": {"num_ctx": QWEN_NUM_CTX, "temperature": 0},
        },
        timeout=600,
    )
    response.raise_for_status()
    data = response.json()
    if data.get("prompt_eval_count", 0) >= QWEN_NUM_CTX - 100:
        print(f"⚠️ Qwen's input was probably cut off ({data['prompt_eval_count']} tokens): increase QWEN_NUM_CTX or send less text")
    answer = re.sub(r"<think>.*?</think>", "", data["message"]["content"], flags=re.DOTALL).strip()
    return {"answer": answer, "cost": 0.0, "seconds": time.time() - start}


def generate_claude(system_prompt, user_message):
    start = time.time()
    response = client_claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    answer = "".join(b.text for b in response.content if b.type == "text")
    if response.stop_reason == "max_tokens":
        answer += "\n\n⚠️ [Answer cut off: the max_tokens limit was reached]"
    return {"answer": answer, "cost": claude_cost(CLAUDE_MODEL, response.usage), "seconds": time.time() - start}


GENERATORS = {"qwen": generate_qwen, "claude": generate_claude}
print("RAG functions ready ✅")

RAG functions ready ✅


## Step 5 — Load the questions

`questions.csv` has one row per question:

| Column | Meaning | Example |
|---|---|---|
| `id` | Short name | `q01` |
| `question` | The question (Portuguese or English) | What percentage of Natura's ingredients are renewable? |
| `company` | Filter for the search (empty = all companies) | `natura` |
| `answerable` | `yes` if the reports contain the answer, `no` for "I don't know" tests | `yes` |
| `expected_answer` | Reference answer, written after checking the PDF | 94.7% of Natura brand ingredients... |
| `expected_sources` | Where the answer is: `company:page`, separated by `;` | `natura:75;natura:18` |
| `verified` | `yes` once the answer was checked in the PDF | `yes` |
| `category` | Type of question, for the per-category scores | `qualifier` |
| `visual` | Where the answer is: `chart`, `infographic`, `table` or `text` (grading only) | `chart` |
| `notes` | Comments, e.g. known inconsistencies in the report | Qualifier: abroad only |

⚠️ An evaluation is only as good as its reference answers, so only questions with `verified = yes` are used.

In [ ]:
import pandas as pd

QUESTIONS_FILE = f"{EVALS_DIR}/questions.csv"
questions = pd.read_csv(QUESTIONS_FILE, dtype=str).fillna("")

skipped = questions[questions["verified"].str.lower() != "yes"]
questions = questions[questions["verified"].str.lower() == "yes"].reset_index(drop=True)

print(f"{len(questions)} verified questions will be evaluated "
      f"({(questions['answerable'] == 'yes').sum()} answerable, {(questions['answerable'] == 'no').sum()} 'I don't know' tests).")
if len(skipped):
    print(f"⚠️ Skipping {len(skipped)} unverified questions: {', '.join(skipped['id'])}")
questions[["id", "question", "company", "answerable", "expected_sources"]]

114 verified questions will be evaluated (106 answerable, 8 'I don't know' tests).


,id,question,company,answerable,expected_sources
0,q01,What percentage of Natura's ingredients are re...,natura,yes,natura:75
1,q02,Qual é o percentual de fórmulas biodegradáveis...,natura,yes,natura:75
2,q03,How much did Natura reduce its absolute carbon...,natura,yes,natura:61;natura:18
3,q04,Qual foi o conteúdo de plástico reciclado nas ...,natura,yes,natura:62;natura:18
4,q05,What was Natura's integrated profit and loss (...,natura,yes,natura:27;natura:18
...,...,...,...,...,...
109,q110,Did Petrobras's absolute operational emissions...,petrobras,yes,petrobras:115
110,q111,Quais eram as emissões operacionais das ativid...,petrobras,yes,petrobras:115
111,q112,What was Petrobras's GHG emissions intensity i...,petrobras,yes,petrobras:115
112,q113,Qual foi a captação de água doce da Petrobras ...,petrobras,yes,petrobras:143


## Step 6 — The graders

**Retrieval grader (code, free):** checks whether an expected page is among the retrieved chunks, and at which position (rank).
For multi-company questions, it also checks **company coverage**: did the chunks include every company the answer needs?

**Citation grader (code, free):** reads the `[1]`, `[2][3]` citations in the answer, finds the pages they point to,
and checks whether any of them is an expected page.

**Answer grader (LLM-as-a-judge):** Claude reads the question, the reference answer and the system's answer, and gives a grade:
`correct` = 1 point · `partial` = 0.5 · `incorrect` = 0.
The rubric separates **key facts** (must be present and correct: numbers, units, qualifiers like "abroad only") from
**extra context** (nice to have), and the judge replies in JSON.

⚠️ **Known limitation:** when the judge is Claude, it may be slightly friendlier to Claude's own answers. Grades were spot-checked by hand.

In [ ]:
import json

def parse_expected(expected_sources):
    '''"natura:75;natura:18" -> {("natura", 75), ("natura", 18)}'''
    pairs = set()
    for item in expected_sources.split(";"):
        item = item.strip()
        if ":" in item:
            company, page = item.split(":")
            pairs.add((company.strip().lower(), int(page)))
    return pairs


def grade_retrieval(chunks, expected):
    '''Returns (hit, rank). rank = position of the first expected page (1 = best), or None.'''
    for rank, c in enumerate(chunks, start=1):
        if (c["company"], c["page"]) in expected:
            return True, rank
    return False, None


def grade_company_coverage(chunks, expected):
    '''Share of the expected companies that appear in the retrieved chunks (1.0 = all companies present).'''
    expected_companies = {company for company, _ in expected}
    retrieved_companies = {c["company"] for c in chunks}
    return len(expected_companies & retrieved_companies) / len(expected_companies)


def grade_citations(answer, chunks, expected):
    '''Did the answer cite at least one chunk from an expected page?'''
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}
    cited_pages = {(chunks[n - 1]["company"], chunks[n - 1]["page"]) for n in cited if 1 <= n <= len(chunks)}
    return len(cited_pages & expected) > 0


JUDGE_PROMPT = '''You are a strict grader for a question-answering system about company sustainability reports.
You will receive a question, a reference answer written by a human expert, and the system's answer.

If ANSWERABLE is "yes":
- "correct": the system answer contains the key facts of the reference, with the same numbers, units, and qualifiers
  (e.g., region such as "abroad", scope, year, or whether a number is part of a total or the total itself).
- "partial": the main fact is right, but a qualifier needed to interpret it correctly is missing or wrong,
  or one part of a multi-part answer (for example, one of the companies being compared) is missing.
- "incorrect": the key fact is missing, wrong, or the system says it couldn't find the answer.

What counts as KEY: the number or fact that directly answers the question, plus the qualifiers that change its meaning
(scope, region, part vs. total, units/currency, and a yes/no conclusion when the question asks for one).
The reference may also include EXTRA CONTEXT (targets, base years, year-over-year changes, comparisons with limits).
Missing extra context alone does NOT lower the grade: if all key facts are right, the grade is "correct".
Ignore extra details that don't contradict the reference. Language differences (Portuguese/English) don't matter.

If ANSWERABLE is "no" (the information is NOT in the reports):
- "correct": the system says it couldn't find the information and does not present invented or unrelated numbers as the answer.
- "incorrect": the system gives an answer as if it were known.

Reply with JSON only, no other text: {"grade": "correct" | "partial" | "incorrect", "reason": "<one short sentence>"}'''

SCORES = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}


def judge(question, reference, answerable, answer):
    response = client_claude.messages.create(
        model=JUDGE_MODEL,
        max_tokens=2048,
        system=JUDGE_PROMPT,
        messages=[{"role": "user", "content":
            f"QUESTION: {question}\n\nANSWERABLE: {answerable}\n\nREFERENCE ANSWER: {reference}\n\nSYSTEM ANSWER: {answer}"}],
    )
    text = "".join(b.text for b in response.content if b.type == "text")
    cost = claude_cost(JUDGE_MODEL, response.usage)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    try:
        result = json.loads(match.group(0))
        grade = result.get("grade", "").lower()
        if grade not in SCORES:
            raise ValueError(grade)
        return grade, result.get("reason", ""), cost
    except Exception:
        # The JSON is sometimes slightly broken (e.g. a stray '"}' at the end),
        # but the grade itself is still readable: pick it out directly
        g = re.search(r'"grade"\s*:\s*"(correct|partial|incorrect)"', text, flags=re.IGNORECASE)
        r = re.search(r'"reason"\s*:\s*"(.*?)"\s*[,}]', text, flags=re.DOTALL)
        if g:
            return g.group(1).lower(), (r.group(1) if r else text[:200]), cost
        return "error", f"Could not parse the judge's reply: {text[:200]}", cost


# Quick self-test of the code graders (no API calls)
_chunks = [{"company": "natura", "page": 12}, {"company": "natura", "page": 75}]
_expected = parse_expected("natura:75;natura:18")
print(grade_retrieval(_chunks, _expected))                      # (True, 2)
print(grade_citations("It is 94.7% [2].", _chunks, _expected))  # True
print(grade_citations("It is 94.7% [1].", _chunks, _expected))  # False

(True, 2)
True
False


## Step 7 — Run the evaluation

For each question, the passages are retrieved **once** and the **same sources** go to every LLM, like `compare()` in `02_ask.ipynb`.
The run stops before starting if Qwen is enabled but Ollama isn't running.

Results are saved to Drive **after every question**, so nothing is lost if Colab disconnects.

⏱️ 114 questions with both models take about 30–40 minutes (Qwen is the slow part).
💲 About US$ 0.17 per run (Claude Haiku answers + judge for both models).

In [ ]:
import json
from datetime import datetime

# Safety check: don't spend half an hour recording 114 connection errors
if "qwen" in LLMS and not ollama_is_running():
    raise RuntimeError("Ollama isn't running, so every Qwen answer would fail. "
                       "Run Step 2a again (or set USE_QWEN = False), then run this cell.")

run_id = datetime.now().strftime("%Y%m%d_%H%M")
results_file = f"{EVALS_DIR}/results_{CONFIG_NAME}_{run_id}.csv"
rows = []

for i, q in questions.iterrows():
    company = q["company"] or None
    expected = parse_expected(q["expected_sources"])
    chunks = get_context(q["question"], n_results=N_RESULTS, company=company)
    user_message = build_user_message(q["question"], chunks)

    hit, rank = grade_retrieval(chunks, expected) if expected else (None, None)
    coverage = grade_company_coverage(chunks, expected) if expected else None

    for llm in LLMS:
        try:
            out = GENERATORS[llm](SYSTEM_PROMPT, user_message)
        except Exception as e:
            out = {"answer": f"ERROR: {e}", "cost": 0.0, "seconds": 0.0}

        if out["answer"].startswith("ERROR:"):
            # A code error is not an answer: don't send it to the judge, and keep it out of the scores
            grade, reason, judge_cost = "error", out["answer"], 0.0
            print(f"⚠️ {q['id']} {llm}: {out['answer']}")
        else:
            grade, reason, judge_cost = judge(q["question"], q["expected_answer"], q["answerable"], out["answer"])
        cites_ok = grade_citations(out["answer"], chunks, expected) if expected else None

        rows.append({
            "config": CONFIG_NAME, "run_id": run_id, "id": q["id"], "llm": llm,
            "question": q["question"], "answerable": q["answerable"], "category": q.get("category", ""), "visual": q.get("visual", ""),
            "grade": grade, "score": SCORES.get(grade),
            "retrieval_hit": hit, "retrieval_rank": rank, "company_coverage": coverage, "citation_ok": cites_ok,
            "seconds": round(out["seconds"], 1), "cost": out["cost"], "judge_cost": judge_cost,
            "answer": out["answer"], "judge_reason": reason, "expected_answer": q["expected_answer"],
            "retrieved": "; ".join(f"{c['company']}:{c['page']}" for c in chunks),
        })
        print(f"[{i + 1}/{len(questions)}] {q['id']} {llm:<6} → {grade:<9} "
              f"(retrieval: {'✅' if hit else '❌' if hit is not None else '—'}, {out['seconds']:.1f} s)")

    pd.DataFrame(rows).to_csv(results_file, index=False)   # save after every question

results = pd.DataFrame(rows)
print("\nSaved:", results_file)

[1/114] q01 qwen   → correct   (retrieval: ✅, 156.7 s)
[1/114] q01 claude → correct   (retrieval: ✅, 1.3 s)
[2/114] q02 qwen   → partial   (retrieval: ✅, 6.2 s)
[2/114] q02 claude → correct   (retrieval: ✅, 1.5 s)
[3/114] q03 qwen   → correct   (retrieval: ✅, 8.4 s)
[3/114] q03 claude → correct   (retrieval: ✅, 1.0 s)
[4/114] q04 qwen   → correct   (retrieval: ✅, 8.3 s)
[4/114] q04 claude → correct   (retrieval: ✅, 2.4 s)
[5/114] q05 qwen   → correct   (retrieval: ✅, 9.6 s)
[5/114] q05 claude → correct   (retrieval: ✅, 1.1 s)
[6/114] q06 qwen   → correct   (retrieval: ✅, 9.3 s)
[6/114] q06 claude → correct   (retrieval: ✅, 2.1 s)
[7/114] q07 qwen   → correct   (retrieval: ✅, 12.2 s)
[7/114] q07 claude → correct   (retrieval: ✅, 1.4 s)
[8/114] q08 qwen   → correct   (retrieval: ✅, 10.6 s)
[8/114] q08 claude → correct   (retrieval: ✅, 1.8 s)
[9/114] q09 qwen   → partial   (retrieval: ✅, 9.6 s)
[9/114] q09 claude → partial   (retrieval: ✅, 2.7 s)
[10/114] q10 qwen   → correct   (retrieval

## Step 8 — Scores

- **answer_score**: average score on answerable questions (1.0 = all correct)
- **refusal_acc**: share of "I don't know" tests handled correctly
- **retrieval_hit**: share of answerable questions with an expected page among the retrieved chunks (the same for every LLM, since they share the sources)
- **citation_ok**: share of answers that cited an expected page
- **multi_company_coverage**: for multi-company questions, the share of expected companies present in the retrieved chunks
  (`retrieval_hit` counts a hit when only **one** of them was found; this metric checks **all** of them)
- **n_errors**: answers that failed because of a code or API error. They are left out of the scores, so a valid run has 0.

In [ ]:
def summarize(df):
    answerable = df[df["answerable"] == "yes"]
    unanswerable = df[df["answerable"] == "no"]
    return pd.Series({
        "answer_score": answerable["score"].mean(),
        "refusal_acc": unanswerable["score"].mean(),
        "retrieval_hit": answerable["retrieval_hit"].astype(float).mean(),
        "citation_ok": answerable["citation_ok"].astype(float).mean(),
        "multi_company_coverage": answerable.loc[answerable["category"] == "multi_company", "company_coverage"].astype(float).mean()
                                  if "company_coverage" in answerable else float("nan"),
        "avg_seconds": df["seconds"].mean(),
        "llm_cost_usd": df["cost"].sum(),
        "judge_cost_usd": df["judge_cost"].sum(),
        "n_questions": len(df),
        "n_errors": (df["grade"] == "error").sum(),
    })

summary = results.groupby("llm").apply(summarize, include_groups=False).round(3)
summary

,answer_score,refusal_acc,retrieval_hit,citation_ok,multi_company_coverage,avg_seconds,llm_cost_usd,judge_cost_usd,n_questions,n_errors
llm,,,,,,,,,,
claude,0.863,1.000,0.821,0.811,1.0,2.618,0.133,0.019,114.0,0.0
qwen,0.764,0.875,0.821,0.651,1.0,12.363,0.000,0.018,114.0,0.0


### Scores by question type

Each question has a `category` (number, qualifier, target, yes_no, multi_part, multi_company, conflict, refusal),
which shows **which kinds of questions** each model struggles with.

In [ ]:
by_category = results.pivot_table(index="category", columns="llm", values="score", aggfunc="mean").round(2)
by_category["n_questions"] = results.groupby("category")["id"].nunique()
by_category

llm,claude,qwen,n_questions
category,,,
conflict,1.00,1.00,2
multi_company,0.90,0.87,15
multi_part,0.79,0.86,7
number,0.87,0.77,35
qualifier,0.93,0.79,14
refusal,1.00,0.88,8
target,0.86,0.77,22
yes_no,0.73,0.45,11


### Scores by visual type

The `visual` column says where the answer is in the report:
`chart` = only in a chart · `infographic` = text and charts mixed · `table` = a text table · `text` = normal text.

This column is **only used for grading**: the RAG never sees it (it receives only the question and the company).
It measures whether the image reader (`USE_VISION` in `01_indexing.ipynb`) helps the visual questions without hurting the others.

In [ ]:
if "visual" in results and results["visual"].ne("").any():
    by_visual = results.pivot_table(index="visual", columns="llm", values="score", aggfunc="mean").round(2)
    by_visual["n_questions"] = results.groupby("visual")["id"].nunique()
    display(by_visual)
else:
    print("No 'visual' column in questions.csv")

llm,claude,qwen,n_questions
visual,,,
chart,0.50,0.25,8
infographic,0.92,0.85,31
table,0.85,0.69,24
text,0.91,0.84,51


## Step 9 — Error analysis

The score shows **how much** is wrong; the errors show **why**. For each wrong answer:
- **Retrieval ❌** → a search problem (chunking, embeddings, top-k…)
- **Retrieval ✅ but the answer is wrong** → a generation problem (model, prompt…)

In [ ]:
errors = results[results["score"] < 1]
for _, r in errors.iterrows():
    rank = int(r["retrieval_rank"]) if pd.notna(r["retrieval_rank"]) else "-"
    print(f"── {r['id']} · {r['llm'].upper()} · {r['grade']} · retrieval: {r['retrieval_hit']} (rank {rank})")
    print("Q:", r["question"])
    print("Expected:", r["expected_answer"])
    print("Answer:", r["answer"][:600])
    print("Judge:", r["judge_reason"])
    print("Retrieved:", r["retrieved"], "\n")

if errors.empty:
    print("No errors! 🎉 Time to add harder questions.")

── q02 · QWEN · partial · retrieval: True (rank 1)
Q: Qual é o percentual de fórmulas biodegradáveis da Natura?
Expected: 97.6% of rinse-off formulas are biodegradable (2030 target: at least 95%).
Answer: O percentual de fórmulas biodegradáveis da Natura é de 97,6% [1].
Judge: The system gives 97.6% but omits the qualifier that this refers to rinse-off formulas specifically, which changes the meaning of the figure.
Retrieved: natura:75; natura:73; natura:62; natura:60; natura:12; natura:76; natura:72; natura:18 

── q09 · QWEN · partial · retrieval: True (rank 1)
Q: Quais foram as emissões operacionais de óleo e gás da Petrobras em 2025?
Expected: 47 million tCO2e from oil and gas activities (excluding thermoelectric power), an increase of 3 million tCO2e vs. the previous year (18% below 2015).
Answer: As emissões operacionais de óleo e gás da Petrobras em 2025 totalizaram 47 milhões de tCO2e [1].
Judge: The system gives the correct 47 million tCO2e figure but omits the scope qualifier

## Step 10 — Experiment history

Every run is saved as `results_<CONFIG_NAME>_<date>.csv`, and this cell summarizes **all** of them.
Each experiment changes **one thing at a time** (with a new `CONFIG_NAME`), so a difference in the scores can be attributed to that change.

In [3]:
import glob

all_runs = pd.concat([pd.read_csv(f) for f in glob.glob(f"{EVALS_DIR}/results_*.csv")], ignore_index=True)
history = all_runs.groupby(["config", "run_id", "llm"]).apply(summarize, include_groups=False).round(3)
history[["answer_score", "refusal_acc", "retrieval_hit", "multi_company_coverage", "citation_ok", "avg_seconds", "llm_cost_usd", "n_questions"]]

answer_score  \
config                              run_id        llm                    
final_D_no_vision                   20261116_1721 claude         0.863   
                                                  qwen           0.764   
final_D_vision                      20261110_1805 claude         0.915   
                                                  qwen           0.816   
per_company_top10_page5_plus_chunks 20261106_0137 claude         0.884   
                                                  qwen           0.768   

                                                          refusal_acc  \
config                              run_id        llm                   
final_D_no_vision                   20261116_1721 claude        1.000   
                                                  qwen          0.875   
final_D_vision                      20261110_1805 claude        1.000   
                                                  qwen          1.000   
per_company_top10_page5_plus_chunks 20261106_0137 claude        1.000   
                                                  qwen          0.875   

                                                          retrieval_hit  \
config                              run_id        llm                     
final_D_no_vision                   20261116_1721 claude          0.821   
                                                  qwen            0.821   
final_D_vision                      20261110_1805 claude          0.887   
                                                  qwen            0.887   
per_company_top10_page5_plus_chunks 20261106_0137 claude          0.798   
                                                  qwen            0.798   

                                                          multi_company_coverage  \
config                              run_id        llm                              
final_D_no_vision                   20261116_1721 claude                     1.0   
                                                  qwen                       1.0   
final_D_vision                      20261110_1805 claude                     1.0   
                                                  qwen                       1.0   
per_company_top10_page5_plus_chunks 20261106_0137 claude                     1.0   
                                                  qwen                       1.0   

                                                          citation_ok  \
config                              run_id        llm                   
final_D_no_vision                   20261116_1721 claude        0.811   
                                                  qwen          0.651   
final_D_vision                      20261110_1805 claude        0.868   
                                                  qwen          0.689   
per_company_top10_page5_plus_chunks 20261106_0137 claude        0.788   
                                                  qwen          0.636   

                                                          avg_seconds  \
config                              run_id        llm                   
final_D_no_vision                   20261116_1721 claude        2.618   
                                                  qwen         12.363   
final_D_vision                      20261110_1805 claude        2.527   
                                                  qwen         13.355   
per_company_top10_page5_plus_chunks 20261106_0137 claude        2.296   
                                                  qwen         11.182   

                                                          llm_cost_usd  \
config                              run_id        llm                    
final_D_no_vision                   20261116_1721 claude         0.133   
                                                  qwen           0.000   
final_D_vision                      20261110_1805 claude         0.148   
                                                  qwen           0

## Summary

An **evaluation pipeline** that turns "it looks good" into numbers:
- Retrieval and generation measured **separately**
- An LLM judge that is strict about numbers, units and qualifiers
- Results saved for every experiment, so changes can be compared

**Experiments run so far:** top-k, per-company search, whole pages vs. chunks, pages + chunks, and the image reader (vision).
The results are summarized in the repository's README.

**Possible next experiments:**
- Hybrid search (BM25 + embeddings) or a reranker, since most remaining misses are retrieval misses
- Prompt rules for Qwen: compare results with targets explicitly, and handle the Portuguese thousands separator (112.744 = 112,744)
- Claude Sonnet as the judge, to measure how much the cheaper judge agrees with a stricter one